# 02. FILIP Expanded Prompt Ensembling & Token Max-Pooled Visualization

This notebook visualizes **FILIP fine-grained cross-modal token-to-patch alignment maps** with:
1. **Clinical Acronym Expansion**: Scans all 23 PTB-XL Subclass diagnostic acronyms and expands them into full clinical descriptions.
2. **Prompt Ensembling**: Averages text projection embeddings across 4 clinical templates per diagnosis for robust zero-shot ranking.
3. **Token Max-Pooling**: Max-pools similarity maps across subword tokens to produce unified diagnostic heatmaps.
4. **Warm Red vs. Transparent Overlay**: Renders clean, clutter-free heatmaps where background regions are fully transparent and high-confidence lead regions glow in warm red.

In [ ]:
import os
import sys
import yaml
import random
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from transformers import CLIPTokenizer
from matplotlib.colors import LinearSegmentedColormap

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from filip.model.filip_ecg_model import FILIPECGModel
from filip.data.dataset import ECGImageDataset

## 1. Full PTB-XL Subclass Clinical Expansion Mapping & Ensemble Templates

In [ ]:
# Mapping dictionary for all 23 PTB-XL Subclass diagnostic acronyms
SUBCLASS_EXPANSIONS = {
    'AMI': 'anterior myocardial infarction',
    'LAFB/LPFB': 'left anterior or posterior fascicular block',
    'LVH': 'left ventricular hypertrophy',
    'STTC': 'ST-T segment changes',
    'IMI': 'inferior myocardial infarction',
    'SEHYP': 'septal hypertrophy',
    'CRBBB': 'complete right bundle branch block',
    'WPW': 'Wolff-Parkinson-White syndrome',
    'LAO/LAE': 'left atrial overload or enlargement',
    'NORM': 'normal sinus rhythm',
    'ISC': 'ischemic ST-T changes',
    'AVB': 'atrioventricular block',
    'RAO/RAE': 'right atrial overload or enlargement',
    'LMI': 'lateral myocardial infarction',
    'ISCI': 'ischemic infarction',
    'ISCA': 'ischemic anterior changes',
    'NST': 'non-specific ST-T changes',
    'CLBBB': 'complete left bundle branch block',
    'ILBBB': 'incomplete left bundle branch block',
    'IRBBB': 'incomplete right bundle branch block',
    'PMI': 'posterior myocardial infarction',
    'RVH': 'right ventricular hypertrophy',
    'IVCD': 'intraventricular conduction delay'
}

# Ensemble templates per diagnosis
ENSEMBLE_TEMPLATES = [
    "an electrocardiogram showing {expansion}",
    "12-lead ECG demonstrating {expansion}",
    "patient ECG with {expansion} diagnosis",
    "diagnostic findings of {expansion}"
]

# Custom Warm Red vs. Transparent Colormap (alpha 0.0 at low sim, 0.75 warm red at peak)
colors = [(1.0, 0.0, 0.0, 0.0), (1.0, 0.15, 0.0, 0.75)]
warm_red_cmap = LinearSegmentedColormap.from_list('WarmRedTransparent', colors)

## 2. Load Model & Tokenizer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to 23 PTB-XL Subclass adaptation config & checkpoint
# Default to newest ViT-Large report-aligned model checkpoint
checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'vit_large_ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_adapt_100')

checkpoint_path = os.path.join(checkpoint_dir, 'best.pt')
if not os.path.exists(checkpoint_path):
    checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pt')

# Select matching config file corresponding to checkpoint architecture
if 'vit_large' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt_vit_large', 'ptbxl_sub_adapt.yaml')
elif 'report_align' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt', 'ptbxl_sub_adapt.yaml')
else:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'ptbxl_sub_adapt.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"Loading config from: {config_path}")

model = FILIPECGModel(config).to(device)
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    sd = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(sd, strict=False)
    print(f"Loaded FILIP model from: {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}. Using base initialized model.")

text_encoder_name = config.get('model', {}).get('text_encoder', 'openai/clip-vit-large-patch14')
tokenizer = CLIPTokenizer.from_pretrained(text_encoder_name)
model.eval()

## 3. Select Sample (Specified Study ID or Random Selection)

In [ ]:
# Set TARGET_STUDY_ID to a specific Study ID (e.g. "12703_hr"), or leave empty "" for random selection
TARGET_STUDY_ID = ""

data_root = os.path.join(PROJECT_ROOT, config.get('data_root', 'data/ptbxl_sub_class/'))
image_size = config.get('model', {}).get('image_size', 224)
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = ECGImageDataset(data_root=data_root, split='test', dataset_name=config.get('dataset_name', 'ptbxl'), transform=transform)

# Select target record index
selected_idx = None
if TARGET_STUDY_ID.strip():
    for idx, record in enumerate(test_dataset.records):
        if str(record.get('study_id')) == TARGET_STUDY_ID.strip():
            selected_idx = idx
            break
    if selected_idx is None:
        print(f"Study ID '{TARGET_STUDY_ID}' not found in test set. Defaulting to random selection.")

if selected_idx is None:
    selected_idx = random.randint(0, len(test_dataset) - 1)

sample = test_dataset[selected_idx]
image_tensor = sample['images'].unsqueeze(0).to(device)
study_id = sample['sample_ids']
diagnosis_targets = sample['diagnosis_targets']

gt_labels = []
if diagnosis_targets is not None:
    for i, target_val in enumerate(diagnosis_targets):
        if target_val > 0.5:
            gt_labels.append(test_dataset.diagnosis_list[i])

print("=" * 60)
print(f"Selected Sample Index: {selected_idx} / {len(test_dataset)}")
print(f"Selected Study ID:     {study_id}")
print(f"Ground Truth Diagnoses: {gt_labels if gt_labels else 'None / Normal'}")
print("=" * 60)

## 4. Compute Prompt Ensembling & Token Max-Pooled Heatmaps

In [ ]:
target_classes = gt_labels if gt_labels else ['NORM']
patch_size = config.get('model', {}).get('patch_size', 14)
grid_size = image_size // patch_size

with torch.no_grad():
    # 1. Vision Patch Embeddings
    patch_features = model.vision_encoder(image_tensor) # [1, P, H]
    if hasattr(model, 'report_alignment_head'):
        img_proj = model.report_alignment_head.image_projection(patch_features) # [1, P, Align_Dim]
    else:
        img_proj = patch_features
    img_proj = F.normalize(img_proj, p=2, dim=-1)[0] # [P, Align_Dim]

    class_heatmaps = {}
    for cls_code in target_classes:
        expansion = SUBCLASS_EXPANSIONS.get(cls_code, cls_code.lower())
        prompts = [tmpl.format(expansion=expansion) for tmpl in ENSEMBLE_TEMPLATES]
        
        # Tokenize and encode all ensemble prompts
        text_inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        text_outputs = model.text_encoder(**text_inputs)
        text_features = text_outputs.last_hidden_state # [Num_Prompts, T, H]
        
        if hasattr(model, 'report_alignment_head'):
            txt_proj = model.report_alignment_head.text_projection(text_features) # [Num_Prompts, T, Align_Dim]
        else:
            txt_proj = text_features
            
        txt_proj = F.normalize(txt_proj, p=2, dim=-1)
        
        # Compute prompt-averaged token similarity map: [T, P]
        # Einstein summation across batch/prompts
        token_sims = torch.einsum('bta,pa->btp', txt_proj, img_proj).mean(dim=0) # [T, P]
        
        # Max pooling across content tokens (skip start/end special tokens)
        valid_t_indices = [i for i, id_val in enumerate(text_inputs['input_ids'][0]) if id_val not in [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id]]
        if valid_t_indices:
            max_pooled_map = token_sims[valid_t_indices].max(dim=0).values.cpu().numpy().reshape(grid_size, grid_size)
        else:
            max_pooled_map = token_sims.max(dim=0).values.cpu().numpy().reshape(grid_size, grid_size)
            
        # Min-max normalization for visualization
        max_pooled_map = (max_pooled_map - max_pooled_map.min()) / (max_pooled_map.max() - max_pooled_map.min() + 1e-8)
        class_heatmaps[cls_code] = max_pooled_map

print(f"Computed Prompt-Ensembled Max-Pooled Heatmaps for: {list(class_heatmaps.keys())}")

## 5. Render Warm Red vs. Transparent Overlay

In [ ]:
raw_img_path = os.path.join(data_root, 'images', f"{study_id}-0.png")
if not os.path.exists(raw_img_path):
    raw_img_path = os.path.join(data_root, 'images', f"{study_id}.png")
raw_image = Image.open(raw_img_path).convert('RGB').resize((image_size, image_size))

num_plots = len(class_heatmaps)
fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))
if num_plots == 1:
    axes = [axes]

for ax_idx, (cls_code, heatmap) in enumerate(class_heatmaps.items()):
    expansion = SUBCLASS_EXPANSIONS.get(cls_code, cls_code)
    
    axes[ax_idx].imshow(raw_image)
    # Render Warm Red vs. Transparent overlay
    im = axes[ax_idx].imshow(heatmap, cmap=warm_red_cmap, extent=(0, image_size, image_size, 0))
    axes[ax_idx].set_title(f"Diagnosis: {cls_code}\n({expansion})", fontsize=11)
    axes[ax_idx].axis('off')
    fig.colorbar(im, ax=axes[ax_idx], fraction=0.046, pad=0.04, label="Activation")

plt.tight_layout()
plt.show()